<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/05_applications/hybrid_search_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Search RAG Pipeline

## Objective

Build a Retrieval-Augmented Generation pipeline that combines
semantic search and keyword search to improve document retrieval
before generating answers.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline
import pandas as pd
import numpy as np

In [ ]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

In [ ]:
documents = [
    "Machine learning tutorials help beginners understand ML concepts.",
    "Deep learning uses neural networks to learn complex patterns.",
    "Natural language processing focuses on understanding text data.",
    "Python is widely used for machine learning and data science.",
    "Transformers such as BERT use attention mechanisms."
]

In [ ]:
doc_embeddings = bi_encoder.encode(documents)

In [ ]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)

In [ ]:
query = "How does BERT attention work?"

query_embedding = bi_encoder.encode(query)

In [ ]:
semantic_scores = cosine_similarity(
    [query_embedding],
    doc_embeddings
)[0]

In [ ]:
query_tfidf = vectorizer.transform([query])

keyword_scores = cosine_similarity(
    query_tfidf,
    tfidf_matrix
)[0]

In [ ]:
hybrid_scores = 0.6 * semantic_scores + 0.4 * keyword_scores

In [17]:
df = pd.DataFrame({
    "Document": documents,
    "Hybrid Score": hybrid_scores
}).sort_values(by="Hybrid Score", ascending=False)

top_k = 3

candidates = df.head(top_k)

In [18]:
pairs = [[query, doc] for doc in candidates["Document"]]

rerank_scores = cross_encoder.predict(pairs)

candidates["Rerank Score"] = rerank_scores

candidates = candidates.sort_values(
    by="Rerank Score",
    ascending=False
)

/tmp/ipykernel_186/227045844.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates["Rerank Score"] = rerank_scores


In [19]:
context = " ".join(candidates["Document"].tolist())

In [20]:
prompt = f"""
Use the context to answer the question in one short paragraph.

Context:
{context}

Question:
{query}

Answer:
"""

In [21]:
response = generator(
    prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.6
)
print(response[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the context to answer the question in one short paragraph.

Context:
Transformers such as BERT use attention mechanisms. Deep learning uses neural networks to learn complex patterns. Natural language processing focuses on understanding text data.

Question:
How does BERT attention work?

Answer:
BERT attention is used to measure the number of words that are used in a given sentence.
Question:
How does BERT focus on the number of words that are used in a given sentence
